In [1]:
# Mount Google Drive (if modules stored there)
from google.colab import drive
drive.mount('/content/drive')

# Add module directory to Python path
import sys
sys.path.append('/content/drive/MyDrive/VISUAL_ASSISTANCE')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


color

In [11]:
%%writefile /content/drive/MyDrive/VISUAL_ASSISTANCE/color_module.py
import cv2
import numpy as np

# Standard HSV ranges
color_ranges = {
    "Pink": [(140, 20, 50), (170, 255, 255)],
    "Magenta": [(125, 20, 50), (150, 255, 255)],
    "Lavender": [(110, 20, 80), (140, 255, 255)],

    "Green": [(35, 50, 50), (85, 255, 255)],
    "White": [(0, 0, 200), (255, 40, 255)],
    "Black": [(0, 0, 0), (180, 255, 40)],
}


def detect_color(image):
    """
    Detects dominant colors in the input image based on HSV ranges.
    Returns a string describing colors that appear more than threshold %.
    """

    # Resize image for consistency
    img = cv2.resize(image, (640, 640))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    color_percentage = {}

    for color, (lower, upper) in color_ranges.items():
        mask = cv2.inRange(hsv, np.array(lower), np.array(upper))
        ratio = (cv2.countNonZero(mask) / (img.size / 3)) * 100
        color_percentage[color] = ratio

    # Prepare readable output
    detected = [f"{color}" for color, perc in color_percentage.items() if perc > 0.2]

    if not detected:
        return "No significant color detected."

    return "Detected colors: " + ", ".join(detected)



Overwriting /content/drive/MyDrive/VISUAL_ASSISTANCE/color_module.py


In [12]:
from color_module import detect_color


In [15]:
import cv2

image_path = "/content/drive/MyDrive/assistive-vision-dataset/raw/ocr/currency/Train/1Hundrednote/1.jpg"
image = cv2.imread(image_path)

answer = detect_color(image)
print(answer)



Detected colors: Pink, Magenta, Lavender, White


vqa

In [17]:
%%writefile /content/drive/MyDrive/VISUAL_ASSISTANCE/vqa_module.py
# vqa_module.py
# Visual Question Answering (LLaVA) module
# Provides: answer_question(image, english_question, max_new_tokens=200)
# Also: unload_model() to free GPU memory

import os
import warnings
from typing import Union

import torch
from PIL import Image
import numpy as np

# Try to import Llava class and processor; the exact import may depend on the package version.
# If your environment provides a different import path adjust accordingly.
try:
    from transformers import AutoProcessor
    from transformers import LlavaForConditionalGeneration
except Exception as e:
    # keep the import error visible but do not crash on import; the model will fail to load later with clear message
    LlavaForConditionalGeneration = None
    AutoProcessor = None
    warnings.warn(f"Could not import Llava classes from transformers here: {e}. "
                  "Make sure package versions are installed in the Colab environment.")

# Module-level cached model & processor
_model = None
_processor = None
_model_id = "llava-hf/llava-1.5-7b-hf"  # change if you want a different LLaVA model


def _pil_from_input(image: Union[np.ndarray, Image.Image]) -> Image.Image:
    """
    Accepts an OpenCV ndarray (BGR) or a PIL Image and returns a PIL Image in RGB mode.
    """
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    if isinstance(image, np.ndarray):
        # OpenCV uses BGR order, convert to RGB
        if image.ndim == 3 and image.shape[2] == 3:
            img_rgb = image[:, :, ::-1]
            return Image.fromarray(img_rgb).convert("RGB")
        else:
            # grayscale or other
            return Image.fromarray(image).convert("RGB")
    raise ValueError("Input image must be a PIL.Image or an OpenCV (numpy) array.")


def _get_device():
    return torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")


def _load_model(device: torch.device = None, model_id: str = None):
    """
    Lazy-load the model and processor into module-level variables.
    Uses 4-bit loading if possible to reduce memory (dependent on bitsandbytes & transformers support).
    """
    global _model, _processor, _model_id

    if model_id:
        _model_id = model_id

    if _model is not None and _processor is not None:
        return _model, _processor

    if AutoProcessor is None or LlavaForConditionalGeneration is None:
        raise ImportError(
            "Required classes (AutoProcessor, LlavaForConditionalGeneration) are not available. "
            "Ensure you installed the correct transformers/llava package in Colab."
        )

    device = device or _get_device()
    print(f"[vqa_module] Loading model {_model_id} on device {device} ... (this may take a while)")

    # Recommended: try low-mem loading settings first (4-bit + device_map='auto'), fallback to safer CPU float16 if fails
    try:
        # Preferred (if environment supports bitsandbytes + 4-bit)
        _model = LlavaForConditionalGeneration.from_pretrained(
            _model_id,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            load_in_4bit=True,
            device_map="auto",
        )
        _processor = AutoProcessor.from_pretrained(_model_id)
        print("[vqa_module] Model loaded with 4-bit + device_map='auto'.")
    except Exception as e:
        warnings.warn(f"4-bit loading failed or not supported in this environment: {e}. "
                      "Falling back to CPU/float16 load (may be slow).")
        # fallback: CPU/float16
        _model = LlavaForConditionalGeneration.from_pretrained(
            _model_id,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto" if torch.cuda.is_available() else None,
        )
        _processor = AutoProcessor.from_pretrained(_model_id)
        print("[vqa_module] Model loaded (fallback).")

    # Move to CPU if device is cpu, or leave device_map manage placement
    if device.type == "cpu":
        _model.to("cpu")

    return _model, _processor


def unload_model():
    """
    Unload the model and free GPU memory. Call this after you finish using the module to avoid memory hogging.
    """
    global _model, _processor
    try:
        if _model is not None:
            del _model
        if _processor is not None:
            del _processor
    except Exception:
        pass
    _model = None
    _processor = None
    # free gpu memory
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    print("[vqa_module] Model and processor unloaded.")


def answer_question(image: Union[np.ndarray, Image.Image],
                    english_question: str,
                    max_new_tokens: int = 200,
                    model_id: str = None) -> str:
    """
    Run LLaVA VQA on the provided image and english_question.
    - image: PIL image or OpenCV ndarray (BGR)
    - english_question: question text in English (master notebook should translate to English first)
    - returns: answer string in English

    This function lazily loads the model on first call and keeps it in memory until unload_model() is called.
    """
    global _model, _processor

    if not isinstance(english_question, str) or len(english_question.strip()) == 0:
        raise ValueError("Please provide a non-empty English question string.")

    # Convert image to PIL
    pil_img = _pil_from_input(image)

    # Load model if needed
    device = _get_device()
    if _model is None or _processor is None:
        _load_model(device=device, model_id=model_id)

    if _model is None or _processor is None:
        raise RuntimeError("Model failed to load. Check logs and package installation in the Colab environment.")

    # Build prompt (this follows your earlier format)
    prompt = f"USER: <image>\n{english_question}\nASSISTANT:"

    # Prepare inputs through processor
    # Important: put tensors to CPU first to avoid some device_map issues, model generation will handle placement
    inputs = _processor(images=pil_img, text=prompt, return_tensors="pt")
    # move inputs to model device if model expects them there (device_map auto usually handles it)
    # But to be safe, we will not force move here; many Llava variants handle CPU tensors with device_map placement.

    # Generate
    _model.eval()
    with torch.no_grad():
        try:
            output = _model.generate(**inputs, max_new_tokens=max_new_tokens)
        except Exception as e:
            # If generation fails due to device mismatch, try moving inputs to model device
            try:
                model_device = next(_model.parameters()).device
                inputs = {k: v.to(model_device) for k, v in inputs.items()}
                output = _model.generate(**inputs, max_new_tokens=max_new_tokens)
            except Exception as ee:
                raise RuntimeError(f"Generation failed: {e} / fallback failed: {ee}")

    # Decode output
    try:
        answer = _processor.decode(output[0], skip_special_tokens=True)
    except Exception:
        # If processor has no decode, try model's tokenizer or raw decode
        try:
            answer = output[0]
            if isinstance(answer, torch.Tensor):
                answer = answer.cpu().numpy().tolist()
            answer = str(answer)
        except Exception:
            answer = "Could not decode model output."

    # Post-process answer
    answer = answer.strip()
    if len(answer) == 0:
        answer = "No answer generated."

    return answer


Overwriting /content/drive/MyDrive/VISUAL_ASSISTANCE/vqa_module.py


In [18]:
from vqa_module import answer_question, unload_model

# image can be OpenCV array or PIL.Image
# english_question should already be translated to English by master notebook.
answer_en = answer_question(image, "What is on the table?")
print("LLaVA answer (EN):", answer_en)

# when done (or to free memory between requests)
unload_model()


`torch_dtype` is deprecated! Use `dtype` instead!


[vqa_module] Loading model llava-hf/llava-1.5-7b-hf on device cpu ... (this may take a while)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

/content/drive/MyDrive/VISUAL_ASSISTANCE/vqa_module.py:88: UserWarning: 4-bit loading failed or not supported in this environment: No package metadata was found for bitsandbytes. Falling back to CPU/float16 load (may be slow).
  warnings.warn(f"4-bit loading failed or not supported in this environment: {e}. "


...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

[vqa_module] Model loaded (fallback).
LLaVA answer (EN): USER:  
What is on the table?
ASSISTANT: There is a purple banknote on the table.
[vqa_module] Model and processor unloaded.


object

In [19]:
%%writefile /content/drive/MyDrive/VISUAL_ASSISTANCE/object_module.py
import torch
from ultralytics import YOLO
import numpy as np
from PIL import Image

# Module-level cached YOLO model
_yolo_model = None
_model_path = "yolov8n.pt"   # Fastest / lightest model


def _pil_from_input(image):
    """
    Accepts an OpenCV ndarray (BGR) or PIL.Image and returns PIL.Image in RGB.
    """
    if isinstance(image, Image.Image):
        return image.convert("RGB")

    if isinstance(image, np.ndarray):
        if image.ndim == 3 and image.shape[2] == 3:
            rgb = image[:, :, ::-1]  # BGR → RGB
            return Image.fromarray(rgb)
        return Image.fromarray(image).convert("RGB")

    raise ValueError("Input must be a PIL.Image or OpenCV ndarray.")


def _load_yolo_model(model_path=None):
    """
    Lazy-load YOLO model and store globally.
    """
    global _yolo_model, _model_path

    if model_path:
        _model_path = model_path

    if _yolo_model is None:
        print("[object_module] Loading YOLOv8 model...")
        _yolo_model = YOLO(_model_path)   # ultralytics handles device placement automatically

    return _yolo_model


def unload_yolo():
    """
    Frees YOLO model from memory (optional).
    """
    global _yolo_model
    try:
        del _yolo_model
        _yolo_model = None
        torch.cuda.empty_cache()
        print("[object_module] YOLO model unloaded.")
    except Exception:
        pass


def run_yolo(image):
    """
    Runs YOLO object detection on a PIL or OpenCV image.
    Returns a text list of detected objects with confidences.
    """

    global _yolo_model

    # Convert image to PIL
    pil_img = _pil_from_input(image)

    # Load YOLO lazily
    model = _load_yolo_model()

    # Run detection
    results = model.predict(pil_img, verbose=False)

    detections = results[0].boxes

    if detections is None or len(detections) == 0:
        return "No objects detected in the image."

    names = model.names  # class names
    output_lines = []

    for box in detections:
        cls = int(box.cls[0])
        conf = float(box.conf[0]) * 100
        label = names.get(cls, f"class_{cls}")
        output_lines.append(f"{label} ({conf:.1f}%)")

    return "Detected objects: " + ", ".join(output_lines)


Overwriting /content/drive/MyDrive/VISUAL_ASSISTANCE/object_module.py


In [22]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.1 MB/s eta 0:00:00


In [24]:
from object_module import run_yolo, unload_yolo
import cv2

import cv2

image = cv2.imread("/content/drive/MyDrive/assistive-vision-dataset/raw/ocr/currency/Train/1Hundrednote/1.jpg")
YOLO_output = run_yolo(image)
print(YOLO_output)



[object_module] Loading YOLOv8 model...
Detected objects: book (52.2%), book (39.3%), book (33.8%)


ocr

In [25]:
%%writefile /content/drive/MyDrive/VISUAL_ASSISTANCE/ocr_module.py
# ocr_module.py
# OCR module using EasyOCR
# Provides: extract_text(image)
#           unload_ocr() to free RAM

import easyocr
import numpy as np
from PIL import Image
import torch

# Cached EasyOCR reader
_reader = None


def _pil_from_input(image):
    """
    Converts OpenCV (BGR) or PIL image into a PIL RGB image.
    """
    if isinstance(image, Image.Image):
        return image.convert("RGB")

    if isinstance(image, np.ndarray):
        if image.ndim == 3 and image.shape[2] == 3:
            rgb = image[:, :, ::-1]   # BGR → RGB
            return Image.fromarray(rgb)
        else:
            return Image.fromarray(image).convert("RGB")

    raise ValueError("Input must be a PIL.Image or OpenCV ndarray.")


def _load_reader(lang_list=['en']):
    """
    Lazy-load EasyOCR reader (loads only once).
    """
    global _reader
    if _reader is None:
        print("[ocr_module] Loading EasyOCR reader...")
        _reader = easyocr.Reader(lang_list)
    return _reader


def unload_ocr():
    """
    Clear the OCR reader from memory (optional).
    """
    global _reader
    try:
        del _reader
        _reader = None
        torch.cuda.empty_cache()
        print("[ocr_module] OCR reader unloaded.")
    except Exception:
        pass


def extract_text(image):
    """
    Runs OCR on an input image (PIL or OpenCV).
    Returns a clean string of detected text.
    """

    global _reader

    pil_img = _pil_from_input(image)

    # Convert back to numpy for easyocr
    img_np = np.array(pil_img)

    # Load EasyOCR lazily
    reader = _load_reader()

    # Perform OCR
    results = reader.readtext(img_np)

    if not results:
        return "No readable text found in the image."

    # Collect text segments
    extracted = []
    for bbox, text, prob in results:
        extracted.append(text)

    if len(extracted) == 0:
        return "No readable text found in the image."

    return " ".join(extracted)


Overwriting /content/drive/MyDrive/VISUAL_ASSISTANCE/ocr_module.py


In [27]:
!pip install easyocr
!pip install opencv-python pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.8/963.8 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 13.9 MB/s eta 0:00:00


In [29]:
from ocr_module import extract_text, unload_ocr
import cv2

image = cv2.imread("/content/drive/MyDrive/assistive-vision-dataset/raw/ocr/download.jpeg")
text_out = extract_text(image)

print(text_out)
unload_ocr()


[ocr_module] Loading EasyOCR reader...
477T Res enveb4nk Qfinc"e 71552 8
[ocr_module] OCR reader unloaded.


currency

In [40]:
%%writefile /content/drive/MyDrive/VISUAL_ASSISTANCE/currency_module.py
import numpy as np
import tensorflow as tf
from PIL import Image
import json
import os

# >>>>>>>  FIXED PATH  <<<<<<<
MODEL_PATH = "/content/drive/MyDrive/VISUAL_ASSISTANCE/currency_model.h5"
CLASS_MAP_PATH = "/content/drive/MyDrive/VISUAL_ASSISTANCE/currency_class_map.json"

_model = None
_class_map = None
IMAGE_SIZE = (224, 224)

def _pil_from_input(image):
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    if isinstance(image, np.ndarray):
        if image.ndim == 3:
            rgb = image[:, :, ::-1]
            return Image.fromarray(rgb).convert("RGB")
        return Image.fromarray(image).convert("RGB")
    raise ValueError("Invalid image type.")

def _load_model():
    global _model
    if _model is None:
        if not os.path.exists(MODEL_PATH):
            raise FileNotFoundError("Currency model not found at " + MODEL_PATH)
        _model = tf.keras.models.load_model(MODEL_PATH)
    return _model

def _load_map():
    global _class_map
    if _class_map is None:
        if not os.path.exists(CLASS_MAP_PATH):
            raise FileNotFoundError("Class map not found at " + CLASS_MAP_PATH)
        with open(CLASS_MAP_PATH, "r") as f:
            _class_map = json.load(f)
    return _class_map

def identify_currency(image):
    model = _load_model()
    class_map = _load_map()

    img = _pil_from_input(image)
    img = img.resize(IMAGE_SIZE)
    arr = np.array(img).astype("float32") / 255.0
    arr = np.expand_dims(arr, 0)

    preds = model.predict(arr)[0]
    idx = int(np.argmax(preds))
    conf = float(preds[idx])

    label = class_map.get(str(idx), f"class_{idx}")

    return label, conf

def unload_currency_model():
    global _model, _class_map
    _model = None
    _class_map = None
    tf.keras.backend.clear_session()
    print("Currency model unloaded.")


Overwriting /content/drive/MyDrive/VISUAL_ASSISTANCE/currency_module.py


In [2]:
from currency_module import identify_currency

import cv2
img = cv2.imread("/content/drive/MyDrive/assistive-vision-dataset/raw/ocr/currency/Test/1Hundrednote/1.jpg")

label, conf = identify_currency(img)
print(label, conf)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 421ms/step
1Hundrednote 0.9999902248382568


for fixing currency module


In [35]:
!ls /content/drive/MyDrive/VISUAL_ASSISTANCE/

color_module.py		 currency_model.h5   object_module.py  __pycache__
currency_class_map.json  currency_module.py  ocr_module.py     vqa_module.py


In [42]:
!find / -name "currency_model.h5"


find: ‘/proc/68/task/68/net’: Invalid argument
find: ‘/proc/68/net’: Invalid argument
/content/drive/MyDrive/VISUAL_ASSISTANCE/currency_model.h5


In [43]:
!sed -n '1,80p' /content/drive/MyDrive/VISUAL_ASSISTANCE/currency_module.py


import numpy as np
import tensorflow as tf
from PIL import Image
import json
import os

# >>>>>>>  FIXED PATH  <<<<<<<
MODEL_PATH = "/content/drive/MyDrive/VISUAL_ASSISTANCE/currency_model.h5"
CLASS_MAP_PATH = "/content/drive/MyDrive/VISUAL_ASSISTANCE/currency_class_map.json"

_model = None
_class_map = None
IMAGE_SIZE = (224, 224)

def _pil_from_input(image):
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    if isinstance(image, np.ndarray):
        if image.ndim == 3:
            rgb = image[:, :, ::-1]
            return Image.fromarray(rgb).convert("RGB")
        return Image.fromarray(image).convert("RGB")
    raise ValueError("Invalid image type.")

def _load_model():
    global _model
    if _model is None:
        if not os.path.exists(MODEL_PATH):
            raise FileNotFoundError("Currency model not found at " + MODEL_PATH)
        _model = tf.keras.models.load_model(MODEL_PATH)
    return _model

def _load_map():
    global _class_map
    if _c